In [3]:
import openmdao.api as om
import numpy as np
from pathlib import Path
import os

from simplerecorder import SimpleRecorder
from mysite import get_setup_params
from aep import AEPCompStochastic, AEPComp
from constraints import SpacingConstraintComp, BoundaryConstraintComp
from aggregator import ConstraintAggregator
from penalty import PenaltyObjectiveComp, FinalRMSViol

from Drivers import NCG, SGD

In [4]:
BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
def build_problem(K=50, csv_filename="default.csv", results_dir=RESULTS_DIR, seed=1):
    params = get_setup_params(csv_filename)

    site = params['site']
    turbine = params['turbine']
    wfm = params['wfm']
    x_init = params['x_init']
    y_init = params['y_init']
    boundary_vertices = params['boundary_vertices']
    min_spacing_d = params['min_spacing_d']
    n_turbines = params['n_turbines']
    D = params['diameter']

    min_spacing_m = min_spacing_d * D

    try:
        aep0 = float(wfm(x_init, y_init).aep().sum())
        print(f"Initial AEP: {aep0:.3f} GWh")
    except Exception as e:
        print(f"Initial AEP error: {e}")
        aep0 = None

    prob = om.Problem()
    m = prob.model

    log_path = results_dir / f"run_seed_{seed}.csv"

    recorder = SimpleRecorder(
        prob,
        out_path=log_path,
        x_name='x',
        y_name='y',
        aep_name='aep_comp_deterministic.aep',
        obj_name='objective',
        iter_name='opt_iter',
        viol_name='rms_viol'
    )
    recorder.start()

    indeps = m.add_subsystem('indeps', om.IndepVarComp(), promotes=['*'])
    indeps.add_output('x', val=x_init, units='m')
    indeps.add_output('y', val=y_init, units='m')
    indeps.add_output('opt_iter', val=0.0)

    aep_comp = AEPCompStochastic(
        wake_model=wfm,
        site=site,
        wt_x=x_init,
        wt_y=y_init,
        aep_ref=1.0,
        recorder=None,
        n_cpu=1,
        K=K,
    )

    m.add_subsystem(
        'aep_comp_deterministic',
        AEPComp(
            wake_model=wfm,
            wt_x=x_init,
            wt_y=y_init,
            aep_ref=1.0,
            n_cpu=1,
        ),
        promotes_inputs=['x', 'y']
    )

    m.add_subsystem(
        'aep_comp',
        aep_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['aep'],
    )

    spacing_comp = SpacingConstraintComp(
        n_turbines=n_turbines,
        min_spacing=min_spacing_m,
        eps=1e-12,
    )
    m.add_subsystem(
        'spacing_comp',
        spacing_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['spacing_cons'],
    )

    boundary_comp = BoundaryConstraintComp(
        boundary_vertices=boundary_vertices,
        n_turbines=n_turbines,
    )
    m.add_subsystem(
        'boundary_comp',
        boundary_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['boundary_cons'],
    )

    agg_comp = ConstraintAggregator(n_turbines=n_turbines)
    m.add_subsystem(
        'constraint_agg',
        agg_comp,
        promotes_inputs=['spacing_cons', 'boundary_cons'],
        promotes_outputs=['g_vector'],
    )

    penalty_comp = PenaltyObjectiveComp(n_constraints=agg_comp.m_total)
    m.add_subsystem(
        'penalty_comp',
        penalty_comp,
        promotes_inputs=['aep', 'g_vector'],
        promotes_outputs=['objective', 'penalty'],
    )

    nc = agg_comp.m_total
    m.add_subsystem(
        'rms_viol',
        FinalRMSViol(nconstraints=nc),
        promotes_outputs=['rms_viol']
    )
    m.connect('g_vector', 'rms_viol.g_vector')

    prob.driver = NCG(maxiter=30)
    prob.driver.options['learning_rate'] = params['diameter'] / 5.0
    prob.driver.options['gamma_min'] = 0.2 * (params['diameter'] / 5.0)
    prob.driver.options['lower'] = 1e-6
    prob.driver.options['upper'] = 1e-1
    prob.driver.options['tol'] = 1e-6
    prob.driver.options['disp'] = True

    m.add_design_var(
        'x',
        lower=boundary_vertices[:, 0].min(),
        upper=boundary_vertices[:, 0].max()
    )
    m.add_design_var(
        'y',
        lower=boundary_vertices[:, 1].min(),
        upper=boundary_vertices[:, 1].max()
    )

    m.add_objective('aep', scaler=-1.0)
    m.add_constraint('penalty', lower=0.0, scaler=1.0)

    return prob, recorder, log_path

In [6]:
def main(csv_filename="default.csv", K=50, seed=10):
    csv_filename = f"100turb_3600m_kdt_{seed}.csv"

    prob, recorder, log_path = build_problem(
        K=K,
        csv_filename=csv_filename,
        results_dir=RESULTS_DIR,
        seed=seed
    )

    prob.setup()
    prob.run_driver()

    return prob, recorder, log_path

In [7]:
prob, recorder, log_path = main(seed=1, K=50)

Loaded: 100turb_3600m_kdt_1.csv (100 turbines)
Boundary: 0m x 3600m
Initial AEP: 496.257 GWh
Optimization SUCCESSFUL.
Current stochastic function value f_t: -568.332547
Iterations: 30
Function evaluations: 30
Gradient evaluations: 30
Gradient infinity norm: 4.234140e-02

--- Optimization result ---
AEP (normalized): [-568.332547]
Constraint violation norm: [0.000000]
Penalized objective: [-568.332547]


In [8]:
print("Log save in:", log_path)
print("AEP final:", prob.get_val('aep'))
print("Penalty final:", prob.get_val('penalty'))
print("RMS viol final:", prob.get_val('rms_viol'))
print("x final:", prob.get_val('x'))
print("y final:", prob.get_val('y'))

Log save in: /Users/brunoboer/Documents/Software/WESL_Optimizer/wesl/Optimizers_Bruno/Results/run_seed_1_scaler.csv
AEP final: [568.3325467]
Penalty final: [29.24382036]
RMS viol final: [0.07609767]
x final: [ 1.59185170e+03  2.78904344e+03 -3.44761858e-01  9.23943497e+02
  4.82336417e+02  8.46078230e+01  6.18520707e+02  1.16303330e+03
  1.46426222e+03  1.70759061e+03  1.56148143e+03  2.48200083e+03
  6.74962071e+02  3.25222187e+03  1.73213226e+01  2.18095372e+03
  1.65564131e+03  2.00945765e+03  2.20432147e+02  8.10700195e+02
  2.99930605e+03  3.60063785e+03  1.22992612e+03  2.57349921e+03
  3.22495157e+03  2.96883823e+03  1.20268728e+02  4.18458628e+02
  4.77048912e+02  3.25573120e+03  3.42869809e+02  1.36801720e+03
  3.34475869e+03  1.76584687e+03  2.69846572e+03  1.27256882e+03
  2.29042783e+03  3.21520632e+03 -9.47222752e-01  2.59351534e+03
  3.58888127e+03  2.81144939e+03  9.06429673e+02  3.10882868e+03
  3.04745746e+02  1.68623908e+03  3.47707342e+03  8.65301050e+02
  1.00028616